# Ноутбук 1: финальная проверка готовой торговой модели

Это финальный ноутбук для проверки уже сохраненной модели. Пользователю достаточно запустить ячейки сверху вниз: ноутбук загрузит данные, очистит их, подготовит признаки, применит модель и покажет понятный сигнал `LONG / SHORT / WAIT`.

В конце есть одна главная тестовая ячейка, которая запускает весь pipeline через функцию `run_model_check()`. Это демонстрация работы готовой модели, а не финансовая рекомендация.


In [1]:

# Если запускаете в Colab и не хватает библиотек, раскомментируйте строку ниже.
# !pip install pandas numpy scikit-learn joblib matplotlib

import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


## Настройки

Все файлы лежат рядом с ноутбуком. Если нужно, пути можно поменять в одном месте.


In [2]:

DATA_PATH = "btcusdt_15m_sample.csv"
MODEL_PATH = "trade_signal_model.joblib"
DEFAULT_LAST_ROWS = 120


## Функции

Код разбит на отдельные функции. В конце есть одна главная функция `run_model_check()`, которую удобно запускать для проверки.


In [3]:

def load_market_file(path=DATA_PATH):
    """Загружает CSV с рыночными данными."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path.resolve()}")
    data = pd.read_csv(path)
    return data


def clean_market_data(raw_data):
    """Приводит таблицу к аккуратному виду: timestamp, сортировка, числовые колонки."""
    data = raw_data.copy()
    if "timestamp" in data.columns:
        data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
        data = data.dropna(subset=["timestamp"]).sort_values("timestamp")
    for col in data.columns:
        if col != "timestamp":
            data[col] = pd.to_numeric(data[col], errors="coerce")
    data = data.reset_index(drop=True)
    return data


def load_model_bundle(path=MODEL_PATH):
    """Загружает сохраненную модель и список признаков."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Файл модели не найден: {path.resolve()}")
    bundle = joblib.load(path)
    required = {"model", "feature_cols", "horizon"}
    missing = required.difference(bundle.keys())
    if missing:
        raise ValueError(f"В файле модели не хватает полей: {sorted(missing)}")
    return bundle


def prepare_features(clean_data, feature_cols):
    """Оставляет только признаки, которые модель ждала при обучении."""
    data = clean_data.copy()
    missing = [col for col in feature_cols if col not in data.columns]
    if missing:
        raise ValueError(
            "В данных не хватает колонок для модели: " + ", ".join(missing[:12])
        )
    features = data[feature_cols].copy()
    return features


def create_loader(features, last_rows=DEFAULT_LAST_ROWS):
    """Создает простой loader: последние N строк для проверки модели."""
    if len(features) == 0:
        raise ValueError("Нет строк для предсказания")
    last_rows = max(1, int(last_rows))
    return features.tail(last_rows).copy()


def predict_market_direction(model, loader):
    """Получает вероятности роста и итоговый класс."""
    if hasattr(model, "predict_proba"):
        proba_up = model.predict_proba(loader)[:, 1]
    else:
        pred = model.predict(loader)
        proba_up = np.asarray(pred, dtype=float)
    signal = np.where(proba_up >= 0.55, "LONG", np.where(proba_up <= 0.45, "SHORT", "WAIT"))
    return pd.DataFrame({"probability_up": proba_up, "signal": signal}, index=loader.index)


def postprocess_prediction(clean_data, predictions):
    """Делает вывод понятным: дата, цена, вероятность и текстовый сигнал."""
    base_cols = []
    for col in ["timestamp", "close", "volume", "rsi14", "rv", "news_sentiment"]:
        if col in clean_data.columns:
            base_cols.append(col)
    result = clean_data.loc[predictions.index, base_cols].copy()
    result = pd.concat([result.reset_index(drop=True), predictions.reset_index(drop=True)], axis=1)
    result["probability_up"] = (result["probability_up"] * 100).round(2)
    return result


def explain_last_signal(result_table, horizon="20 candles / about 5 hours"):
    """Печатает короткий человеческий вывод по последней строке."""
    last = result_table.iloc[-1]
    signal = last["signal"]
    prob = last["probability_up"]
    close = last.get("close", np.nan)
    timestamp = last.get("timestamp", "последняя строка")

    if signal == "LONG":
        text = "модель видит перевес в сторону роста"
    elif signal == "SHORT":
        text = "модель видит перевес в сторону снижения"
    else:
        text = "модель не видит нормального перевеса, лучше пропустить вход"

    print("Последняя проверенная свеча:", timestamp)
    if pd.notna(close):
        print(f"Цена close: {close:,.2f}")
    print(f"Горизонт прогноза: {horizon}")
    print(f"Вероятность роста: {prob:.2f}%")
    print(f"Итоговый сигнал: {signal} — {text}.")


def run_model_check(data_path=DATA_PATH, model_path=MODEL_PATH, last_rows=DEFAULT_LAST_ROWS):
    """Главная функция: загрузка, очистка, подготовка, прогноз и понятный вывод."""
    raw_data = load_market_file(data_path)
    clean_data = clean_market_data(raw_data)
    bundle = load_model_bundle(model_path)
    features = prepare_features(clean_data, bundle["feature_cols"])
    loader = create_loader(features, last_rows=last_rows)
    predictions = predict_market_direction(bundle["model"], loader)
    result = postprocess_prediction(clean_data, predictions)
    explain_last_signal(result, horizon=bundle.get("horizon", "20 candles"))
    return result


## Тестовая ячейка

Можно просто запустить. По умолчанию берется файл `btcusdt_15m_sample.csv`, который лежит рядом.

Если запускаете в Google Colab и хотите проверить свой CSV, можно загрузить файл через кнопку выбора файла.


In [4]:

def choose_file_or_default(default_path=DATA_PATH):
    """В Colab позволяет загрузить свой CSV, локально просто вернет файл по умолчанию."""
    try:
        from google.colab import files
        print("Можно загрузить свой CSV. Если не нужно — просто остановите выбор файла и используйте default_path.")
        uploaded = files.upload()
        if uploaded:
            return next(iter(uploaded.keys()))
    except Exception:
        pass
    return default_path

# Главный запуск проверки модели.
# Если нужен свой файл в Colab, замените DATA_PATH на choose_file_or_default().
result = run_model_check(data_path=DATA_PATH, model_path=MODEL_PATH, last_rows=120)
result.tail(10)


Последняя проверенная свеча: 2026-01-18 04:30:00+00:00
Цена close: 95,074.60
Горизонт прогноза: 20 candles / about 5 hours
Вероятность роста: 44.74%
Итоговый сигнал: SHORT — модель видит перевес в сторону снижения.


,timestamp,close,volume,rsi14,rv,news_sentiment,probability_up,signal
110,2026-01-17 15:15:00+00:00,95465.0,256.571,70.114943,0.002863,0.0,41.32,SHORT
111,2026-01-17 16:45:00+00:00,95254.0,299.533,44.931460,0.003293,0.0,47.83,WAIT
112,2026-01-17 18:15:00+00:00,95409.1,104.836,49.143018,0.003347,0.0,44.07,SHORT
113,2026-01-17 19:45:00+00:00,95247.2,82.411,35.765290,0.002936,0.0,48.99,WAIT
114,2026-01-17 21:00:00+00:00,95300.0,64.832,40.771188,0.002711,0.0,48.56,WAIT
115,2026-01-17 22:30:00+00:00,95085.0,600.768,28.058755,0.003122,0.0,50.72,WAIT
116,2026-01-18 00:00:00+00:00,95133.8,262.614,39.096548,0.002966,0.0,50.50,WAIT
117,2026-01-18 01:30:00+00:00,94937.3,130.288,36.390785,0.003339,0.0,54.12,WAIT
118,2026-01-18 03:00:00+00:00,95017.4,105.787,42.979452,0.003472,0.0,50.89,WAIT
119,2026-01-18 04:30:00+00:00,95074.6,119.436,64.692062,0.003500,0.0,44.74,SHORT


## Как читать результат

- `LONG` — модель считает, что шанс роста выше выбранного порога.
- `SHORT` — модель считает, что шанс снижения выше выбранного порога.
- `WAIT` — перевеса нет, сделку лучше пропустить.

Я специально оставил зону `WAIT`, потому что на рынке много шума. Для учебного проекта важнее показать полный корректный pipeline, чем обещать невозможную точность.
